In [1]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, make_scorer

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

### 1. Загрузка и предобработка

In [2]:
### 1. Загрузка и предобработка

df = pd.read_csv('data/bank_data_train.csv', index_col=0)

x = df.drop(columns=['TARGET'])
y = df['TARGET']

print(f'Balanced: {y.value_counts(normalize=True).values}')

# Удаление колонок с пропусками

missing = x.isna().mean().sort_values(ascending=False)
drop_columns = missing[missing > 0.4].index
x = x.drop(drop_columns, axis=1)
print("Dropped columns:", drop_columns.tolist())

# Преобразование данных
numeric_columns = x.select_dtypes(include='number').columns
category_columns = x.select_dtypes(include='object').columns
print(f'Numeric_columns: {len(numeric_columns)}, Category_columns: {len(category_columns)}')

X = x.copy()

imputer = SimpleImputer(missing_values=np.nan, strategy='median')
X[numeric_columns] = imputer.fit_transform(x[numeric_columns])

scaler = StandardScaler()
X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_values = encoder.fit_transform(x[category_columns])

encoder_df = pd.DataFrame(encoder_values, columns=encoder.get_feature_names_out(), index=X.index)

X = X.drop(columns=category_columns)
X = pd.concat([X, encoder_df], axis=1)

print(X.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

Balanced: [0.91856471 0.08143529]
Dropped columns: ['CLNT_SALARY_VALUE', 'LDEAL_YQZ_COM', 'LDEAL_YQZ_CHRG', 'AVG_PCT_MONTH_TO_PCLOSE', 'MAX_PCLOSE_DATE', 'LDEAL_AMT_MONTH', 'AVG_PCT_DEBT_TO_DEAL_AMT', 'LDEAL_YQZ_PC', 'LDEAL_TENOR_MAX', 'DEAL_YQZ_IR_MIN', 'LDEAL_TENOR_MIN', 'LDEAL_DELINQ_PER_MAXYQZ', 'DEAL_YQZ_IR_MAX', 'LDEAL_USED_AMT_AVG_YQZ', 'MED_DEBT_PRC_YQZ', 'CLNT_JOB_POSITION_TYPE', 'APP_CAR', 'APP_DRIVING_LICENSE', 'APP_TRAVEL_PASS', 'APP_KIND_OF_PROP_HABITATION', 'APP_POSITION_TYPE', 'APP_REGISTR_RGN_CODE', 'CNT_TRAN_CLO_TENDENCY1M', 'SUM_TRAN_CLO_TENDENCY1M', 'APP_COMP_TYPE', 'APP_EMP_TYPE', 'APP_EDUCATION', 'APP_MARITAL_STATUS', 'CNT_TRAN_MED_TENDENCY1M', 'SUM_TRAN_MED_TENDENCY1M', 'CLNT_TRUST_RELATION', 'DEAL_GRACE_DAYS_ACC_AVG', 'DEAL_GRACE_DAYS_ACC_MAX', 'DEAL_GRACE_DAYS_ACC_S1X1', 'SUM_TRAN_AUT_TENDENCY1M', 'CNT_TRAN_AUT_TENDENCY1M', 'LDEAL_ACT_DAYS_ACC_PCT_AVG', 'LDEAL_ACT_DAYS_PCT_TR3', 'LDEAL_ACT_DAYS_PCT_TR', 'LDEAL_ACT_DAYS_PCT_TR4', 'LDEAL_ACT_DAYS_PCT_CURR', 'LDEAL

### 2. Обучение моделей

In [8]:
result = pd.DataFrame(columns=['Model', 'Params', 'Accuracy', 'ROC AUC'])
result = pd.read_csv('data/result.csv', index_col=0)

def write_results(model_name, model, y_true, y_pred, y_pred_proba):
    try:
        params = model.get_params()
    except Exception:
        params = 'None'

    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)

    print(f'Accuracy {accuracy}')
    print(f'ROC AUC {roc_auc}')

    global result

    if model_name in result['Model'].values:
        result.loc[result['Model'] == model_name, ['Params', 'Accuracy', 'ROC AUC']] = [params, accuracy, roc_auc]
    else:
        result = pd.concat([result, pd.DataFrame([{
            'Model': model_name,
            'Params': params,
            'Accuracy': accuracy,
            'ROC AUC': roc_auc
        }])], ignore_index=True)


In [215]:
# Наивный классификатор
classifier = DummyClassifier(strategy='most_frequent')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
y_pred_proba = classifier.predict_proba(X_test)[:, 1]
write_results('DummyClassifier', classifier, y_test, y_pred, y_pred_proba)

Accuracy 0.918564711844365
ROC AUC 0.5


In [228]:
# RandomForest
params = {
    'class_weight': ['balanced', None]
}

classifier = RandomForestClassifier(random_state=21)
grid = RandomizedSearchCV(estimator=classifier, param_distributions=params, n_jobs=-1, verbose=2, random_state=21,
                          n_iter=2, cv=2)

grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)
y_pred_proba = grid.predict_proba(X_test)[:, 1]
write_results('RandomForestClassifier', grid.best_estimator_, y_test, y_pred, y_pred_proba)

Accuracy 0.919071482868324
ROC AUC 0.8235020325977787


In [114]:
# MLPClassifier
params = {
    'solver': ['adam'],
    'alpha': [1e-4, 5e-4, 10e-4, 20e-4],
}

auc_scorer = make_scorer(roc_auc_score, needs_proba=True)

mlp = MLPClassifier(random_state=21, activation='relu', hidden_layer_sizes=(128, 64), max_iter=100, early_stopping=True, n_iter_no_change=20, verbose=True)

grid = RandomizedSearchCV(
    estimator=mlp,  # модель, которую хотим перебрать
    param_distributions=params,  # параметры для перебора
    n_iter=4,  # число случайных комбинаций
    cv=2,  # кросс-валидация
    n_jobs=-1,
    verbose=2,
    random_state=21,
    scoring=auc_scorer
)

grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)
y_pred_proba = grid.predict_proba(X_test)[:, 1]
write_results('MLPClassifier', grid.best_estimator_, y_test, y_pred, y_pred_proba)

Fitting 2 folds for each of 4 candidates, totalling 8 fits
Iteration 1, loss = 0.26629451
Validation score: 0.918497
Iteration 2, loss = 0.24345542
Validation score: 0.918743
Iteration 3, loss = 0.23746209
Validation score: 0.918708
Iteration 4, loss = 0.23377068
Validation score: 0.918391
Iteration 5, loss = 0.23145335
Validation score: 0.918673
Iteration 6, loss = 0.22905477
Validation score: 0.919130
Iteration 7, loss = 0.22746891
Validation score: 0.918356
Iteration 8, loss = 0.22605702
Validation score: 0.918708
Iteration 9, loss = 0.22486371
Validation score: 0.918813
Iteration 10, loss = 0.22341857
Validation score: 0.916983
Iteration 11, loss = 0.22257910
Validation score: 0.918708
Iteration 12, loss = 0.22154322
Validation score: 0.918004
Iteration 13, loss = 0.22054744
Validation score: 0.917722
Iteration 14, loss = 0.21943164
Validation score: 0.918813
Iteration 15, loss = 0.21868637
Validation score: 0.918285
Iteration 16, loss = 0.21795197
Validation score: 0.917758
Iterat

In [256]:
# Keras
input_dim = X_train.shape[1]

keras = Sequential([
    Dense(128, activation='relu', input_dim=input_dim),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

keras.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

keras.fit(X_train, y_train, epochs=100, batch_size=512, verbose=1)
y_pred_proba = keras.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
write_results('Keras', keras, y_test, y_pred, y_pred_proba)

Epoch 1/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9155 - auc: 0.6475 - loss: 0.2831
Epoch 2/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9185 - auc: 0.7256 - loss: 0.2588
Epoch 3/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9185 - auc: 0.7500 - loss: 0.2520
Epoch 4/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9186 - auc: 0.7653 - loss: 0.2473
Epoch 5/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9185 - auc: 0.7750 - loss: 0.2442
Epoch 6/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9186 - auc: 0.7840 - loss: 0.2413
Epoch 7/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9187 - auc: 0.7904 - loss: 0.2392
Epoch 8/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9188 - auc: 0.7954 - loss: 0.2374
Epoch 9/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9189 - auc: 0.7993 - loss: 0.2361
Epoch 10/100
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9189 - auc: 0.8032 - lo

In [22]:
# TensorFlow
class TFMLP:
    def __init__(self, input_dim):
        self.W1 = tf.Variable(tf.random.truncated_normal([input_dim, 128], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([128]))

        self.W2 = tf.Variable(tf.random.truncated_normal([128, 64], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([64]))

        self.W3 = tf.Variable(tf.random.truncated_normal([64, 1], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([1]))

        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def forward(self, X, training=True):

        x = tf.matmul(X, self.W1) + self.b1
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=0.3)
        x = tf.matmul(x, self.W2) + self.b2
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=0.3)
        x = tf.matmul(x, self.W3) + self.b3
        y_hat = tf.sigmoid(x)
        return y_hat

    def compute_loss(self, y_true, y_pred):
        return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))

    def train_step(self, X_batch, y_batch):
        with tf.GradientTape() as tape:
            y_pred = self.forward(X_batch, training=True)
            loss = self.compute_loss(y_batch, y_pred)
        grads = tape.gradient(loss, [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3])
        self.optimizer.apply_gradients(zip(grads, [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]))
        return loss

    def predict_proba(self, X):
        return self.forward(X, training=False).numpy()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

In [32]:
X_train_tf = tf.convert_to_tensor(X_train.values, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train.values.reshape(-1, 1), dtype=tf.float32)

X_test_tf = tf.convert_to_tensor(X_test.values, dtype=tf.float32)
y_test_tf = y_test.values

batch_size = 2048
epochs = 100

dataset_tf = tf.data.Dataset.from_tensor_slices((X_train_tf, y_train_tf))
dataset_tf = dataset_tf.shuffle(buffer_size=X_train.shape[0]).batch(batch_size)

tfmpl = TFMLP(input_dim=X_train.shape[1])

patience = 10
best_auc = 0
epochs_without_improve = 0
best_weights = None

for epoch in range(epochs):
    for X_batch, y_batch in dataset_tf:
        tfmpl.train_step(X_batch, y_batch)

    y_pred_proba = tfmpl.predict_proba(X_test_tf)
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"Epoch {epoch + 1}: ROC AUC = {auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        epochs_without_improve = 0
        best_weights = [
            tf.identity(tfmpl.W1),
            tf.identity(tfmpl.b1),
            tf.identity(tfmpl.W2),
            tf.identity(tfmpl.b2),
            tf.identity(tfmpl.W3),
            tf.identity(tfmpl.b3),
        ]
    else:
        epochs_without_improve += 1

    if epochs_without_improve >= patience:
        print("Early stopping")
        break

    if best_weights is not None:
        tfmpl.W1.assign(best_weights[0])
        tfmpl.b1.assign(best_weights[1])
        tfmpl.W2.assign(best_weights[2])
        tfmpl.b2.assign(best_weights[3])
        tfmpl.W3.assign(best_weights[4])
        tfmpl.b3.assign(best_weights[5])

y_pred = tfmpl.predict(X_test_tf)
y_pred_proba = tfmpl.predict_proba(X_test_tf)
write_results('TFMLP', tfmpl, y_test_tf, y_pred, y_pred_proba)

Epoch 1: ROC AUC = 0.6671
Epoch 2: ROC AUC = 0.7087
Epoch 3: ROC AUC = 0.7288
Epoch 4: ROC AUC = 0.7428
Epoch 5: ROC AUC = 0.7546
Epoch 6: ROC AUC = 0.7645
Epoch 7: ROC AUC = 0.7720
Epoch 8: ROC AUC = 0.7782
Epoch 9: ROC AUC = 0.7837
Epoch 10: ROC AUC = 0.7866
Epoch 11: ROC AUC = 0.7919
Epoch 12: ROC AUC = 0.7956
Epoch 13: ROC AUC = 0.7997
Epoch 14: ROC AUC = 0.8019
Epoch 15: ROC AUC = 0.8044
Epoch 16: ROC AUC = 0.8052
Epoch 17: ROC AUC = 0.8079
Epoch 18: ROC AUC = 0.8100
Epoch 19: ROC AUC = 0.8114
Epoch 20: ROC AUC = 0.8131
Epoch 21: ROC AUC = 0.8147
Epoch 22: ROC AUC = 0.8164
Epoch 23: ROC AUC = 0.8150
Epoch 24: ROC AUC = 0.8157
Epoch 25: ROC AUC = 0.8160
Epoch 26: ROC AUC = 0.8158
Epoch 27: ROC AUC = 0.8163
Epoch 28: ROC AUC = 0.8160
Epoch 29: ROC AUC = 0.8168
Epoch 30: ROC AUC = 0.8176
Epoch 31: ROC AUC = 0.8181
Epoch 32: ROC AUC = 0.8187
Epoch 33: ROC AUC = 0.8200
Epoch 34: ROC AUC = 0.8174
Epoch 35: ROC AUC = 0.8206
Epoch 36: ROC AUC = 0.8200
Epoch 37: ROC AUC = 0.8214
Epoch 38: 

In [58]:
# NumPy
class NumpyMLP:
    def __init__(self, input_dim, lr=0.01):
        self.lr = lr

        self.W1 = np.random.randn(input_dim, 128) * 0.1
        self.b1 = np.zeros((1, 128))

        self.W2 = np.random.randn(128, 64) * 0.1
        self.b2 = np.zeros((1, 64))

        self.W3 = np.random.randn(64, 1) * 0.1
        self.b3 = np.zeros((1, 1))

    def relu(self, X):
        return np.maximum(0, X)

    def sigmoid(self, X):
        return 1 / (1 + np.exp(-X))

    def relu_deriv(self, X):
        return (X > 0).astype(float)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.relu(self.z2)

        self.z3 = self.a2 @ self.W3 + self.b3
        self.y_hat = self.sigmoid(self.z3)

        return self.y_hat

    def compute_loss(self, y_true, y_pred):
        eps = 1e-8
        loss = -(y_true * np.log(y_pred + eps) +
                 (1 - y_true) * np.log(1 - y_pred + eps))
        return np.mean(loss)

    def backward(self, X, y):
        m = X.shape[0]

        dz3 = self.y_hat - y
        dW3 = (self.a2.T @ dz3) / m
        db3 = np.mean(dz3, axis=0, keepdims=True)

        da2 = dz3 @ self.W3.T
        dz2 = da2 * self.relu_deriv(self.z2)

        dW2 = (self.a1.T @ dz2) / m
        db2 = np.mean(dz2, axis=0, keepdims=True)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.relu_deriv(self.z1)

        dW1 = (X.T @ dz1) / m
        db1 = np.mean(dz1, axis=0, keepdims=True)

        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train_step(self, X, y):
        y_pred = self.forward(X)
        loss = self.compute_loss(y, y_pred)
        self.backward(X, y)

        return loss

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

In [61]:
numpymlp = NumpyMLP(input_dim=X_train.shape[1], lr=0.1)

patience = 10
best_auc = 0
epochs_without_improve = 0
best_weights = None

batch_size = 2048
epochs = 100

X_train_np = X_train.to_numpy(dtype=float)
y_train_np = y_train.to_numpy(dtype=float).reshape(-1, 1)
X_test_np = X_test.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float).reshape(-1, 1)

for epoch in range(epochs):

    indices = np.random.permutation(len(X_train))
    X_train_sh = X_train_np[indices]
    y_train_sh = y_train_np[indices].reshape(-1, 1)

    for i in range(0, len(X_train_sh), batch_size):
        X_batch = X_train_sh[i:i + batch_size]
        y_batch = y_train_sh[i:i + batch_size]

        loss = numpymlp.train_step(X_batch, y_batch)

    y_pred = numpymlp.predict_proba(X_test_np)
    auc = roc_auc_score(y_test, y_pred)

    print(f"Epoch {epoch + 1}: AUC = {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        epochs_without_improve = 0
        best_weights = [
            numpymlp.W1.copy(),
            numpymlp.b1.copy(),
            numpymlp.W2.copy(),
            numpymlp.b2.copy(),
            numpymlp.W3.copy(),
            numpymlp.b3.copy(),
        ]
    else:
        epochs_without_improve += 1

    if epochs_without_improve >= patience:
        print("Early stopping")
        break

    if best_weights is not None:
        numpymlp.W1 = best_weights[0]
        numpymlp.b1 = best_weights[1]

        numpymlp.W2 = best_weights[2]
        numpymlp.b2 = best_weights[3]

        numpymlp.W3 = best_weights[4]
        numpymlp.b3 = best_weights[5]

y_pred = numpymlp.predict(X_test_tf)
y_pred_proba = numpymlp.predict_proba(X_test_tf)
write_results('NumpyMLP', numpymlp, y_test_np, y_pred, y_pred_proba)

Epoch 1: AUC = 0.6042
Epoch 2: AUC = 0.6577
Epoch 3: AUC = 0.6830
Epoch 4: AUC = 0.6984
Epoch 5: AUC = 0.7089
Epoch 6: AUC = 0.7171
Epoch 7: AUC = 0.7235
Epoch 8: AUC = 0.7288
Epoch 9: AUC = 0.7340
Epoch 10: AUC = 0.7386
Epoch 11: AUC = 0.7425
Epoch 12: AUC = 0.7456
Epoch 13: AUC = 0.7491
Epoch 14: AUC = 0.7515
Epoch 15: AUC = 0.7536
Epoch 16: AUC = 0.7561
Epoch 17: AUC = 0.7582
Epoch 18: AUC = 0.7605
Epoch 19: AUC = 0.7624
Epoch 20: AUC = 0.7639
Epoch 21: AUC = 0.7648
Epoch 22: AUC = 0.7665
Epoch 23: AUC = 0.7679
Epoch 24: AUC = 0.7691
Epoch 25: AUC = 0.7705
Epoch 26: AUC = 0.7707
Epoch 27: AUC = 0.7720
Epoch 28: AUC = 0.7731
Epoch 29: AUC = 0.7741
Epoch 30: AUC = 0.7757
Epoch 31: AUC = 0.7760
Epoch 32: AUC = 0.7775
Epoch 33: AUC = 0.7781
Epoch 34: AUC = 0.7790
Epoch 35: AUC = 0.7793
Epoch 36: AUC = 0.7800
Epoch 37: AUC = 0.7791
Epoch 38: AUC = 0.7815
Epoch 39: AUC = 0.7821
Epoch 40: AUC = 0.7832
Epoch 41: AUC = 0.7838
Epoch 42: AUC = 0.7845
Epoch 43: AUC = 0.7857
Epoch 44: AUC = 0.78

In [69]:
result.loc[result['Model'] == 'Keras', 'Params'] = [
    {'layers': [(X_train.shape[1], 128), (128, 64), (64, 1)], 'activation': 'relu', 'dropout': 0.3, 'optimizer': 'adam',
     'learning_rate': 0.001, 'loss': 'binary_crossentropy', 'batch_size': 512, 'epochs': 100}]

result.loc[result['Model'] == 'TFMLP', 'Params'] = [
    {'input_dim': X_train.shape[1], 'hidden_layers': [128, 64], 'activation': 'relu', 'dropout': 0.3,
     'optimizer': 'adam', 'learning_rate': 0.001, 'loss': 'binary_crossentropy', 'batch_size': 2048, 'epochs': 100,
     'early_stopping': True, 'patience': 10}]

result.loc[result['Model'] == 'NumpyMLP', 'Params'] = [
    {'input_dim': X_train.shape[1], 'hidden_layers': [128, 64], 'activation': 'relu', 'learning_rate': 0.1,
     'batch_size': 2048, 'epochs': 100, 'early_stopping': True, 'patience': 10}]


In [29]:
# Pytorch
class TorchMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.25)

    def forward(self, X):
        X = F.relu(self.fc1(X))
        X = self.dropout(X)
        X = F.relu(self.fc2(X))
        X = self.dropout(X)
        X = self.out(X)
        return X

    def predict_proba(self, X):
        return torch.sigmoid(self.forward(X))
    
    def predict(self, X, threshold=0.5):
        return self.predict_proba(X) > threshold


In [35]:
patience = 10
best_auc = 0
epochs_without_improve = 0
best_weights = None

batch_size = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Преобразуем данные в тензоры
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1).to(device)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Инициализация модели
torchmlp = TorchMLP(input_dim=X_train.shape[1]).to(device)
optimizer = optim.Adam(torchmlp.parameters(), lr=0.0001)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) 

# Обучение
epochs = 200
for epoch in range(epochs):
    torchmlp.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = torchmlp(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    # Валидация
    torchmlp.eval()
    with torch.no_grad():
        y_pred_test = torchmlp(X_test_tensor).cpu().numpy()
    auc = roc_auc_score(y_test, y_pred_test)
    print(f"Epoch {epoch + 1}: ROC AUC = {auc:.4f}")

    # Early stopping
    if auc > best_auc:
        best_auc = auc
        epochs_without_improve = 0
        best_weights = torchmlp.state_dict()
    else:
        epochs_without_improve += 1

    if epochs_without_improve >= patience:
        print("Early stopping")
        torchmlp.load_state_dict(best_weights)
        break

Device: cuda
Epoch 1: ROC AUC = 0.6865
Epoch 2: ROC AUC = 0.7193
Epoch 3: ROC AUC = 0.7311
Epoch 4: ROC AUC = 0.7401
Epoch 5: ROC AUC = 0.7479
Epoch 6: ROC AUC = 0.7532
Epoch 7: ROC AUC = 0.7570
Epoch 8: ROC AUC = 0.7608
Epoch 9: ROC AUC = 0.7640
Epoch 10: ROC AUC = 0.7666
Epoch 11: ROC AUC = 0.7686
Epoch 12: ROC AUC = 0.7712
Epoch 13: ROC AUC = 0.7728
Epoch 14: ROC AUC = 0.7748
Epoch 15: ROC AUC = 0.7775
Epoch 16: ROC AUC = 0.7788
Epoch 17: ROC AUC = 0.7806
Epoch 18: ROC AUC = 0.7829
Epoch 19: ROC AUC = 0.7843
Epoch 20: ROC AUC = 0.7856
Epoch 21: ROC AUC = 0.7877
Epoch 22: ROC AUC = 0.7888
Epoch 23: ROC AUC = 0.7901
Epoch 24: ROC AUC = 0.7911
Epoch 25: ROC AUC = 0.7929
Epoch 26: ROC AUC = 0.7938
Epoch 27: ROC AUC = 0.7944
Epoch 28: ROC AUC = 0.7960
Epoch 29: ROC AUC = 0.7971
Epoch 30: ROC AUC = 0.7981
Epoch 31: ROC AUC = 0.7990
Epoch 32: ROC AUC = 0.7999
Epoch 33: ROC AUC = 0.8006
Epoch 34: ROC AUC = 0.8017
Epoch 35: ROC AUC = 0.8025
Epoch 36: ROC AUC = 0.8036
Epoch 37: ROC AUC = 0.80

In [36]:
torchmlp.eval()
with torch.no_grad():
    y_pred = torchmlp.predict(X_test_tensor).cpu().numpy()
    y_pred_proba = torchmlp.predict_proba(X_test_tensor).cpu().numpy()
    write_results('PytorchMLP', torchmlp, y_test, y_pred, y_pred_proba)

Accuracy 0.7031166417973479
ROC AUC 0.8321448842144519


In [37]:
result.loc[result['Model'] == 'PytorchMLP', 'Params'] = [
    {'input_dim': X_train.shape[1], 'hidden_layers': [128, 64], 'activation': 'relu', 'learning_rate': 0.0001,
     'batch_size': 512, 'epochs': 200, 'early_stopping': True, 'patience': 10}]

result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.919071,0.823502
2,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch...",0.919043,0.807041
3,Keras,"{'layers': [(55, 128), (128, 64), (64, 1)], 'a...",0.919240,0.826216
4,TFMLP,"{'input_dim': 55, 'hidden_layers': [128, 64], ...",0.919240,0.823400
5,NumpyMLP,"{'input_dim': 55, 'hidden_layers': [128, 64], ...",0.917819,0.803346
6,PytorchMLP,"{'input_dim': 55, 'hidden_layers': [128, 64], ...",0.703117,0.832145


In [38]:
result.to_csv('data/result.csv')
torch.save(torchmlp.state_dict(), 'data/torchmlp_weights.pth')

In [46]:
df_test = pd.read_csv('data/bank_data_test.csv', index_col=0)

all_columns = numeric_columns.to_list() + category_columns.to_list()
x = df_test.drop(columns=['TARGET'])[all_columns]
y = df_test['TARGET']

# Преобразование данных
numeric_columns = x.select_dtypes(include='number').columns
category_columns = x.select_dtypes(include='object').columns
print(f'Numeric_columns: {len(numeric_columns)}, Category_columns: {len(category_columns)}')

X = x.copy()

imputer = SimpleImputer(missing_values=np.nan, strategy='median')
X[numeric_columns] = imputer.fit_transform(x[numeric_columns])

scaler = StandardScaler()
X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_values = encoder.fit_transform(x[category_columns])

encoder_df = pd.DataFrame(encoder_values, columns=encoder.get_feature_names_out(), index=X.index)

X = X.drop(columns=category_columns)
X = pd.concat([X, encoder_df], axis=1)

X = X.reindex(columns=X_train.columns, fill_value=0)

print(X.shape)

Numeric_columns: 43, Category_columns: 1
(88798, 55)


In [57]:
X_test_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)

with torch.no_grad():
    y_pred = torchmlp.predict(X_test_tensor).cpu().numpy()
    y_pred_proba = torchmlp.predict_proba(X_test_tensor).cpu().numpy()
    
pred_df = pd.DataFrame(y_pred_proba, columns=['TARGET'], index=X.index)
pred_df.to_csv('data/predict.csv')